In [ ]:
!pip install -q torch pandas scikit-learn datasets transformers accelerate

In [ ]:
import pandas as pd
import numpy as np
import torch

from datasets import Dataset
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, classification_report

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    set_seed
)

set_seed(42)

In [ ]:
train_df = pd.read_csv("PS_train.csv")
dev_df   = pd.read_csv("PS_dev.csv")
test_df  = pd.read_csv("PS_test_without_labels.csv")

print("Train shape:", train_df.shape)
print("Dev shape:", dev_df.shape)

In [ ]:
label_names = sorted(train_df["labels"].unique())
print("Labels:", label_names)
print("Num labels:", len(label_names))

assert len(label_names) == 7, "Expected 7 classes"

label2id = {l:i for i,l in enumerate(label_names)}
id2label = {i:l for l,i in label2id.items()}

train_df["label_id"] = train_df["labels"].map(label2id)
dev_df["label_id"]   = dev_df["labels"].map(label2id)

In [ ]:
cw = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_df["label_id"]),
    y=train_df["label_id"]
)

class_weights = torch.tensor(cw, dtype=torch.float32)
print("Class weights:", class_weights)
print("Weight shape:", class_weights.shape)

In [ ]:
MODEL_NAME = "xlm-roberta-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id
)

In [ ]:
def tokenize(batch):
    return tokenizer(
        batch["content"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

train_ds = Dataset.from_pandas(
    train_df[["content","label_id"]]
    .rename(columns={"label_id":"labels"})
)

dev_ds = Dataset.from_pandas(
    dev_df[["content","label_id"]]
    .rename(columns={"label_id":"labels"})
)

train_ds = train_ds.map(tokenize, batched=True, remove_columns=["content"])
dev_ds   = dev_ds.map(tokenize, batched=True, remove_columns=["content"])

train_ds.set_format("torch")
dev_ds.set_format("torch")

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "macro_f1": f1_score(labels, preds, average="macro")
    }

In [ ]:
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        loss_fn = torch.nn.CrossEntropyLoss(
            weight=class_weights.to(logits.device)
        )

        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",

    # training
    num_train_epochs=8,
    learning_rate=3e-5,
    weight_decay=0.01,

    # batch size
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,

    # logging
    logging_steps=50,
    report_to="none"
)


In [ ]:
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=dev_ds,
    compute_metrics=compute_metrics
)

trainer.train()

In [ ]:
preds = trainer.predict(dev_ds)
y_pred = np.argmax(preds.predictions, axis=1)

print("\nClassification Report (DEV):")
print(classification_report(
    dev_df["label_id"],
    y_pred,
    target_names=label_names
))

macro = f1_score(dev_df["label_id"], y_pred, average="macro")
print("Macro F1:", macro)

In [ ]:
test_ds = Dataset.from_pandas(test_df[["content"]])
test_ds = test_ds.map(tokenize, batched=True, remove_columns=["content"])
test_ds.set_format("torch")

test_preds = trainer.predict(test_ds)
test_ids = np.argmax(test_preds.predictions, axis=1)

test_df["label"] = [id2label[i] for i in test_ids]

In [ ]:
test_df[["label"]].to_csv("Codeblitz_political.csv", index=False)
print("Submission file created: Codeblitz_political.csv")
